# Análisis de sentimiento de reseñas | WeLoveReviews

## Objetivo de negocio
La account manager necesita estimar cuántas de las 500 reseñas son positivas, neutrales o negativas, comparar la lectura del texto con el promedio humano de 4.5 estrellas y localizar desacuerdos que merezcan revisión. La predicción del modelo es una señal para priorizar análisis, no una verdad sobre la experiencia del cliente.

## Fuente y alcance de la EDA
El dataset disponible es `data/raw/reviews.csv`. El README de la actividad menciona `PROMPT.ts.md`, pero ese archivo no está presente en el repositorio; por eso la EDA se guía por los requisitos entregados y por las columnas reales del CSV. Antes de modelar se comprobarán tamaño, esquema, faltantes, duplicados, distribución de ratings y longitud del texto.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / "data/raw/reviews.csv").exists():
    ROOT = ROOT.parent
INPUT_PATH = ROOT / "data" / "raw" / "reviews.csv"
OUTPUT_PATH = ROOT / "data" / "processed" / "reviews_with_sentiment.csv"

reviews = pd.read_csv(INPUT_PATH)
required_columns = {"review_id", "rating", "review_text"}
missing_columns = required_columns.difference(reviews.columns)
if missing_columns:
    raise ValueError(f"Faltan columnas requeridas: {sorted(missing_columns)}")

reviews["rating"] = pd.to_numeric(reviews["rating"], errors="raise")
if not reviews["rating"].between(1, 5).all():
    raise ValueError("Hay ratings fuera del intervalo de 1 a 5")
if reviews["review_text"].isna().any() or reviews["review_text"].str.strip().eq("").any():
    raise ValueError("Hay reseñas vacías; no se pueden enviar al modelo")

print(f"Filas: {len(reviews)} | Columnas: {list(reviews.columns)}")
print(f"IDs duplicados: {reviews['review_id'].duplicated().sum()} | Textos duplicados: {reviews['review_text'].duplicated().sum()}")
print(f"Rating medio humano: {reviews['rating'].mean():.2f} estrellas")
print("Valores faltantes por columna:")
display(reviews.isna().sum().rename("faltantes").to_frame())
print("Distribución de ratings humanos:")
display(reviews["rating"].value_counts().sort_index().rename_axis("rating").to_frame("reseñas"))
reviews["text_length"] = reviews["review_text"].str.split().str.len()
print("Longitud del texto (palabras):")
display(reviews["text_length"].describe().to_frame())
display(reviews[["review_id", "rating", "review_text"]].head(5))

Filas: 500 | Columnas: ['review_id', 'rating', 'review_text']
IDs duplicados: 0 | Textos duplicados: 7
Rating medio humano: 4.50 estrellas
Valores faltantes por columna:


,faltantes
review_id,0
rating,0
review_text,0


Distribución de ratings humanos:


,reseñas
rating,
1,12
2,18
3,38
4,72
5,360


Longitud del texto (palabras):


,text_length
count,500.000000
mean,29.104000
std,4.824078
min,14.000000
25%,26.000000
50%,29.000000
75%,32.000000
max,42.000000


,review_id,rating,review_text
0,1,5,Visited Harbor House Café last weekend. Great ...
1,2,5,Tried Harbor House Café after seeing it recomm...
2,3,5,Tried Harbor House Café after seeing it recomm...
3,4,5,Visited Harbor House Café last weekend. The st...
4,5,5,Grabbed a quick coffee at Harbor House Café th...


## Plan de modelado
Usaremos `nlptown/bert-base-multilingual-uncased-sentiment`, ya afinado para clasificar texto en una escala de 1 a 5 estrellas. No se entrenará ni ajustará. La revisión del repositorio se fija a un commit inmutable; agruparemos 1–2 como Negativo, 3 como Neutral y 4–5 como Positivo.

**Límite importante:** el modelo se entrenó con reseñas de productos, mientras que estas reseñas describen servicios. El vocabulario, las expectativas y los motivos de satisfacción pueden diferir; por ello inspeccionaremos desacuerdos y no interpretaremos el resultado como una etiqueta objetiva del cliente.

In [2]:
from transformers import pipeline

MODEL_ID = "nlptown/bert-base-multilingual-uncased-sentiment"
MODEL_REVISION = "8f6f4e3a8f70be4b65d3a4a8762b6d781cda240d"

# Se carga una sola vez y procesa el conjunto en lotes.
sentiment_model = pipeline(
    "sentiment-analysis",
    model=MODEL_ID,
    revision=MODEL_REVISION,
    device=-1,
)
raw_predictions = sentiment_model(
    reviews["review_text"].astype(str).tolist(),
    batch_size=16,
    truncation=True,
)

def stars_from_label(label):
    try:
        stars = int(label.split()[0])
    except (IndexError, ValueError) as error:
        raise ValueError(f"Etiqueta inesperada del modelo: {label!r}") from error
    if stars not in range(1, 6):
        raise ValueError(f"Puntuación inesperada del modelo: {stars}")
    return stars

def sentiment_band(stars):
    if stars <= 2:
        return "Negativo"
    if stars == 3:
        return "Neutral"
    return "Positivo"

reviews["predicted_stars"] = [stars_from_label(item["label"]) for item in raw_predictions]
reviews["predicted_sentiment"] = reviews["predicted_stars"].map(sentiment_band)
assert len(reviews) == len(raw_predictions) == 500
print(f"Modelo: {MODEL_ID} @ {MODEL_REVISION}")
print(f"Predicciones asociadas: {reviews['predicted_stars'].notna().sum()} de {len(reviews)}")
display(reviews[["review_id", "rating", "predicted_stars", "predicted_sentiment", "review_text"]].head(8))

/workspaces/Hugo-VC-machine-learning-python-template/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device set to use cpu


Modelo: nlptown/bert-base-multilingual-uncased-sentiment @ 8f6f4e3a8f70be4b65d3a4a8762b6d781cda240d
Predicciones asociadas: 500 de 500


,review_id,rating,predicted_stars,predicted_sentiment,review_text
0,1,5,5,Positivo,Visited Harbor House Café last weekend. Great ...
1,2,5,5,Positivo,Tried Harbor House Café after seeing it recomm...
2,3,5,5,Positivo,Tried Harbor House Café after seeing it recomm...
3,4,5,5,Positivo,Visited Harbor House Café last weekend. The st...
4,5,5,5,Positivo,Grabbed a quick coffee at Harbor House Café th...
5,6,5,5,Positivo,Stopped by Harbor House Café for the first tim...
6,7,5,5,Positivo,Stopped by Harbor House Café for the first tim...
7,8,5,5,Positivo,Came to Harbor House Café for a birthday brunc...


## Resultados y desacuerdos
La distribución del modelo resume el tono estimado del texto; el rating humano resume la valoración explícita. Compararemos ambas medidas con el promedio humano de 4.5 estrellas. Después revisaremos falsos negativos (rating humano 4–5 y predicción 1–2) y desacuerdos con ratings neutrales. Las diferencias son señales para inspección, no una prueba de que el cliente “realmente” sintiera otra cosa.

La revisión manual será una muestra estratificada de 20 reseñas (hasta cuatro por cada rating humano), con el texto y ambas puntuaciones visibles. Así se observan también ratings poco frecuentes, sin presentar esta lectura cualitativa como una evaluación estadística.

In [3]:
sentiment_order = ["Negativo", "Neutral", "Positivo"]
distribution = reviews["predicted_sentiment"].value_counts().reindex(sentiment_order, fill_value=0)
summary = pd.DataFrame({
    "reseñas": distribution,
    "porcentaje": (distribution / len(reviews) * 100).round(1),
})
model_mean = reviews["predicted_stars"].mean()
human_mean = reviews["rating"].mean()
false_negatives = reviews.loc[
    reviews["rating"].ge(4) & reviews["predicted_stars"].le(2),
    ["review_id", "rating", "predicted_stars", "predicted_sentiment", "review_text"],
]
neutral_low_scores = reviews.loc[
    reviews["rating"].eq(3) & reviews["predicted_stars"].le(2),
    ["review_id", "rating", "predicted_stars", "predicted_sentiment", "review_text"],
]

print("Distribución estimada por el modelo:")
display(summary)
print(f"Promedio humano: {human_mean:.2f} estrellas | promedio predicho: {model_mean:.2f} | diferencia: {model_mean - human_mean:+.2f}")
print(f"Falsos negativos (rating humano 4–5, predicción 1–2): {len(false_negatives)}")
display(false_negatives.head(12))
print(f"Ratings neutrales (3) predichos como negativos (1–2): {len(neutral_low_scores)}")
display(neutral_low_scores.head(8))

manual_review = (
    reviews.groupby("rating", group_keys=False)
    .sample(n=4, random_state=42)
    .sort_values(["rating", "review_id"])
    [["review_id", "rating", "predicted_stars", "predicted_sentiment", "review_text"]]
    .reset_index(drop=True)
)
print("Muestra para revisión manual estratificada (20 reseñas):")
with pd.option_context("display.max_colwidth", 360, "display.max_rows", 25):
    display(manual_review)

output_columns = [column for column in reviews.columns if column != "text_length"]
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
reviews[output_columns].to_csv(OUTPUT_PATH, index=False)
print(f"CSV enriquecido escrito: {OUTPUT_PATH} ({len(reviews)} filas)")

Distribución estimada por el modelo:


,reseñas,porcentaje
predicted_sentiment,,
Negativo,47,9.4
Neutral,41,8.2
Positivo,412,82.4


Promedio humano: 4.50 estrellas | promedio predicho: 4.40 | diferencia: -0.10
Falsos negativos (rating humano 4–5, predicción 1–2): 16


,review_id,rating,predicted_stars,predicted_sentiment,review_text
13,14,4,1,Negativo,Tried Harbor House Café after seeing it recomm...
14,15,5,1,Negativo,Every dish was bursting with flavor. The place...
72,73,5,1,Negativo,Stopped by Harbor House Café for the first tim...
86,87,5,1,Negativo,Tried Harbor House Café after seeing it recomm...
137,138,5,2,Negativo,Tried Harbor House Café after seeing it recomm...
199,200,5,2,Negativo,Stopped by Harbor House Café for the first tim...
203,204,5,1,Negativo,Came to Harbor House Café for a birthday brunc...
253,254,5,1,Negativo,Stopped by Harbor House Café for the first tim...
269,270,5,1,Negativo,Stopped by Harbor House Café for the first tim...
321,322,5,1,Negativo,Every dish was bursting with flavor. The staff...


Ratings neutrales (3) predichos como negativos (1–2): 3


,review_id,rating,predicted_stars,predicted_sentiment,review_text
214,215,3,2,Negativo,Came to Harbor House Café for a birthday brunc...
310,311,3,2,Negativo,Tried Harbor House Café after seeing it recomm...
479,480,3,2,Negativo,Visited Harbor House Café last weekend. Averag...


Muestra para revisión manual estratificada (20 reseñas):


,review_id,rating,predicted_stars,predicted_sentiment,review_text
0,78,1,2,Negativo,Regular customer at Harbor House Café here. The portions were way too small for the price. Nobody checked on us the entire meal. Not sure I'll be returning.
1,342,1,2,Negativo,Regular customer at Harbor House Café here. The coffee tasted burnt. Nobody checked on us the entire meal. The place felt cramped and chaotic. Expected a lot more given the reviews.
2,428,1,2,Negativo,Regular customer at Harbor House Café here. The coffee tasted burnt. The staff seemed annoyed to be there. It was way too loud to hold a conversation. Disappointing experience overall.
3,442,1,2,Negativo,Visited Harbor House Café last weekend. The coffee tasted burnt. The cashier was rude when i asked about the menu. The place felt cramped and chaotic. Not sure I'll be returning.
4,148,2,2,Negativo,Tried Harbor House Café after seeing it recommended online. The food was bland and overpriced. Our server got our order wrong twice. Disappointing experience overall.
5,195,2,1,Negativo,Grabbed a quick coffee at Harbor House Café this morning. The food was bland and overpriced. Our server got our order wrong twice. Probably won't recommend it to friends.
6,206,2,2,Negativo,Came to Harbor House Café for a birthday brunch. My eggs came out cold. The cashier was rude when i asked about the menu. Expected a lot more given the reviews.
7,339,2,2,Negativo,Regular customer at Harbor House Café here. The coffee tasted burnt. Nobody checked on us the entire meal. Disappointing experience overall.
8,43,3,3,Neutral,"Visited Harbor House Café last weekend. The coffee was okay. Service was fine, nothing memorable. It's fine for a quick stop."
9,162,3,3,Neutral,"Visited Harbor House Café last weekend. The menu is fine but a bit repetitive. Service was fine, nothing memorable. It's fine for a quick stop."


CSV enriquecido escrito: /workspaces/Hugo-VC-machine-learning-python-template/data/processed/reviews_with_sentiment.csv (500 filas)


## Lectura de resultados y recomendación
De las 500 reseñas, el modelo estimó **412 positivas (82.4%)**, **41 neutrales (8.2%)** y **47 negativas (9.4%)**. El rating humano medio es **4.50**; la media de estrellas predicha es **4.40**, una diferencia de **−0.10**. La diferencia entre escalas es pequeña en promedio, pero no elimina errores individuales ni prueba que la etiqueta predicha refleje la experiencia real.

El dataset tiene 360 ratings de 5, 72 de 4, 38 de 3, 18 de 2 y 12 de 1. No hay valores faltantes ni IDs repetidos; hay 7 textos duplicados y se conservaron porque tienen IDs distintos y no existe una regla de negocio que justifique eliminar reseñas. La longitud media es de 29 palabras. La muestra manual estratificada (20 casos) coincidió en 19 con la banda derivada del rating humano; una reseña neutral (ID 480) fue clasificada negativa. Es una inspección orientativa, no una validación representativa.

El análisis encontró **16 falsos negativos** (ratings 4–5 y modelo 1–2), equivalentes a 3.7% de las 432 reseñas con rating 4–5. Por ejemplo, las reseñas 14 y 73 tienen ratings de 4 y 5, respectivamente, pero predicción de 1 estrella; sus textos contienen señales explícitamente favorables como “Every dish was bursting with flavor”, “The team clearly cares about the customers” y “Service was fast and friendly”. También hay 3 ratings neutrales (3) que el modelo puntuó con 2 estrellas; la reseña 480 menciona sándwiches promedio y personal aparentemente molesto, una mezcla que puede explicar el resultado.

**Hipótesis y acción:** hay desajuste de dominio (reseñas de productos frente a servicios), y el tono mixto o menciones de comida/atención pueden confundir al clasificador. Sin embargo, algunos falsos negativos contienen lenguaje inequívocamente positivo, de modo que el desajuste no basta para explicar cada caso. Recomiendo presentar la distribución como una primera señal, revisar esos 16 textos y validar con etiquetas humanas antes de tomar decisiones sobre el cliente o el personal.

La inferencia de producción queda en `src/app.py`; el CSV conserva los campos de origen y agrega `predicted_stars` y `predicted_sentiment` en `data/processed/reviews_with_sentiment.csv`.

## Extensión opcional: comparación con otro modelo
Como segundo modelo probaremos `tabularisai/multilingual-sentiment-analysis` en estas mismas 500 reseñas. Fijamos la revisión `eea032081f8d247b4303ef3565e7cec1b6f201c9` y mapeamos sus etiquetas `Very Negative`, `Negative`, `Neutral`, `Positive`, `Very Positive` a 1–5 estrellas. Usamos el rating humano 4–5 como referencia positiva para comparar la tasa de falsos negativos; los ratings son una referencia imperfecta, no una etiqueta infalible del sentimiento.

La ficha declara licencia `CC-BY-NC-4.0`; este dato importa antes de integrar el modelo en un trabajo para un cliente comercial.

In [4]:
CANDIDATE_MODEL_ID = "tabularisai/multilingual-sentiment-analysis"
CANDIDATE_REVISION = "eea032081f8d247b4303ef3565e7cec1b6f201c9"
label_to_stars = {
    "Very Negative": 1,
    "Negative": 2,
    "Neutral": 3,
    "Positive": 4,
    "Very Positive": 5,
}

candidate_model = pipeline(
    "sentiment-analysis",
    model=CANDIDATE_MODEL_ID,
    revision=CANDIDATE_REVISION,
    device=-1,
)
candidate_outputs = candidate_model(
    reviews["review_text"].astype(str).tolist(),
    batch_size=16,
    truncation=True,
)
reviews["candidate_stars"] = [label_to_stars[item["label"]] for item in candidate_outputs]
reviews["candidate_sentiment"] = reviews["candidate_stars"].map(sentiment_band)

human_bands = reviews["rating"].map(sentiment_band)
baseline_bands = reviews["predicted_stars"].map(sentiment_band)
candidate_bands = reviews["candidate_stars"].map(sentiment_band)
positive_ratings = reviews["rating"].ge(4)
baseline_false_negatives = positive_ratings & reviews["predicted_stars"].le(2)
candidate_false_negatives = positive_ratings & reviews["candidate_stars"].le(2)

comparison = pd.DataFrame(
    {
        "falsos_negativos (4–5 estrellas)": [
            int(baseline_false_negatives.sum()),
            int(candidate_false_negatives.sum()),
        ],
        "tasa de falsos negativos": [
            baseline_false_negatives.mean() / positive_ratings.mean(),
            candidate_false_negatives.mean() / positive_ratings.mean(),
        ],
        "acierto de banda vs. rating": [
            baseline_bands.eq(human_bands).mean(),
            candidate_bands.eq(human_bands).mean(),
        ],
    },
    index=["Modelo base", "TabularisAI"],
)
comparison["tasa de falsos negativos"] = comparison["tasa de falsos negativos"].map(lambda value: f"{value:.2%}")
comparison["acierto de banda vs. rating"] = comparison["acierto de banda vs. rating"].map(lambda value: f"{value:.1%}")
print(f"Reseñas procesadas por el candidato: {len(candidate_outputs)}")
print("Distribución de etiquetas del candidato:")
display(pd.Series([item["label"] for item in candidate_outputs]).value_counts().rename("reseñas").to_frame())
print("Comparación contra el mismo rating humano:")
display(comparison)
print("Falsos negativos que siguen presentes en el candidato:")
display(
    reviews.loc[
        candidate_false_negatives,
        ["review_id", "rating", "candidate_stars", "review_text"],
    ]
)

Device set to use cpu


Reseñas procesadas por el candidato: 500
Distribución de etiquetas del candidato:


,reseñas
Positive,422
Negative,44
Very Positive,22
Neutral,11
Very Negative,1


Comparación contra el mismo rating humano:


,falsos_negativos (4–5 estrellas),tasa de falsos negativos,acierto de banda vs. rating
Modelo base,16,3.70%,95.0%
TabularisAI,7,1.62%,90.6%


Falsos negativos que siguen presentes en el candidato:


,review_id,rating,candidate_stars,review_text
138,139,5,2,Tried Harbor House Café after seeing it recomm...
219,220,5,2,Visited Harbor House Café last weekend. They w...
230,231,5,2,They went out of their way to accommodate my a...
253,254,5,2,Stopped by Harbor House Café for the first tim...
289,290,5,2,Grabbed a quick coffee at Harbor House Café th...
351,352,5,1,They went out of their way to accommodate my a...
414,415,5,2,Came to Harbor House Café for a birthday brunc...


### Recomendación
En esta comparación, TabularisAI redujo los falsos negativos de ratings 4–5 de **16/432 (3.70%)** a **7/432 (1.62%)**, una reducción relativa del 56.3%. Sin embargo, el acuerdo de la banda de sentimiento con el rating humano bajó de **95.0% a 90.6%**. Los siete falsos negativos restantes son reseñas de 5 estrellas que incluyen que el café “went out of their way to accommodate my allergy”; es una señal de un fallo sistemático que conviene revisar, no un caso aislado.

**No recomiendo sustituir ahora el modelo base en producción.** El candidato mejora el objetivo específico de reducir falsos negativos, pero empeora el acuerdo global y la referencia disponible son ratings, no etiquetas humanas del sentimiento del texto. Además, la ficha del modelo declara licencia **CC-BY-NC-4.0**, que no permite asumir su uso en un servicio comercial sin confirmar autorización. Lo conservaría como experimento; solo reconsideraría el cambio tras resolver la licencia y validar con reseñas etiquetadas por personas, priorizando explícitamente el coste de los falsos negativos.